In [ ]:
import torch

# 전체 흐름: 토큰 ID → 임베딩 x → q, k, v로 투영 → 내적 점수 → 스케일링 → softmax
# 이 예제는 토큰 ID를 임베딩으로 바꾸는 과정을 생략하고, 입력 벡터 x를 직접 지정한다.
# inputs의 각 행은 토큰 하나의 벡터: (토큰 수 6, 입력 차원 3).
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your      (x^1)
    [0.55, 0.87, 0.66], # journey   (x^2)
    [0.57, 0.85, 0.64], # starts    (x^3)
    [0.22, 0.58, 0.33], # with      (x^4)
    [0.77, 0.25, 0.10], # one       (x^5)
    [0.05, 0.89, 0.55], # step      (x^6)
])

In [ ]:
# 1. 두 번째 토큰 journey가 각 토큰을 얼마나 참고할지 계산한다.

x_2 = inputs[1]         # x^2: 두 번째 토큰의 입력 벡터, shape (3,)
d_in = inputs.shape[1]  # 입력 벡터의 차원: 3
d_out = 2               # 이 예제에서 q, k, v 각각의 차원: 2

In [7]:
# 2. 입력 벡터를 서로 다른 역할의 벡터로 바꾸는 세 가중치 행렬을 만든다.
# W_query, W_key, W_value는 각각 (3, 2)이며, 모든 토큰에 같은 행렬을 적용한다.
# 실제 학습에서는 W를 업데이트하지만, 여기서는 계산 연습을 위해 requires_grad=False로 둔다.
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.randn(d_in, d_out), requires_grad=False)

# 같은 x^2를 세 행렬에 각각 투영한다: (3,) @ (3, 2) → (2,).
query_2 = x_2 @ W_query  # q^2: 이 토큰이 어떤 정보를 찾는지 표현
key_2 = x_2 @ W_key      # k^2: 다른 query와 비교할 이 토큰의 특징
value_2 = x_2 @ W_value  # v^2: 가중합할 때 실제로 가져올 이 토큰의 정보

print(f"query_2: {query_2}")
print(f"key_2: {key_2}")
print(f"value_2: {value_2}")

torch.Size([3, 2])
query_2: tensor([-1.1729, -0.0048])
key_2: tensor([-0.1142, -0.7676])
value_2: tensor([0.4107, 0.6274])


In [ ]:
# 3. q^2가 비교할 모든 key와, 나중에 섞을 모든 value를 준비한다.
# 소문자 k^j는 토큰 하나의 벡터, 대문자 K는 모든 k^j를 행으로 쌓은 행렬이다.
# 여기서는 keys가 K, values가 V에 해당한다. 두 번째 토큰만 계산하므로 q는 q^2만 있으면 된다.
keys = inputs @ W_key     # K: (6, 3) @ (3, 2) → (6, 2)
values = inputs @ W_value # V: (6, 3) @ (3, 2) → (6, 2)

print(keys.shape)
print(values.shape)

torch.Size([6, 2])
torch.Size([6, 2])


In [18]:
keys_2 = keys[1]  # k^2: 위에서 구한 key_2와 같은 벡터

# 4. 먼저 점수 하나만 계산한다: score_22 = q^2 · k^2 (자기 자신에 대한 점수).
attn_scores_22 = query_2.dot(keys_2)
print(attn_scores_22)

# print(query_2 @ keys_2)  # 1차원 벡터끼리의 @는 위의 dot과 같은 내적이다.

tensor(0.1376)


In [ ]:
# 5. q^2와 모든 k^j를 각각 내적해 점수 6개를 한 번에 구한다.
# [q^2·k^1, q^2·k^2, ..., q^2·k^6]: 기준 위치 i=2는 고정, 비교 위치 j는 1~6.
# keys.T는 각 key를 열로 배치한다: (2,) @ (2, 6) → (6,).
attn_scores_2 = query_2 @ keys.T

print(query_2) # q^2 하나: shape (2,)
print(keys) # (6, 2)
print(attn_scores_2)

tensor([-1.1729, -0.0048])
tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.6467, -0.6476],
        [ 0.3553, -0.3493]])
tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.4151])


In [ ]:
# 6. 내적 점수 / √d_k → softmax → 어텐션 가중치 순서로 계산한다.
# d_k는 key 벡터 하나의 차원(2)이다. 토큰 수(6)나 입력 차원(3)이 아니다.
# 차원이 커질 때 내적 값이 커져 softmax가 지나치게 쏠리는 것을 √d_k로 나눠 완화한다.
# softmax는 exp(각 점수) / sum(exp(점수들)): 단순히 점수의 합으로 나누는 것과 다르다.

d_k = keys.shape[-1]

# dim=-1: 마지막 축, 즉 비교 대상 토큰 6개에 걸쳐 정규화한다.
attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim=-1)
print(attn_weights_2) # tensor([0.1709, 0.1615, 0.1656, 0.1416, 0.2511, 0.1093])

tensor([0.1709, 0.1615, 0.1656, 0.1416, 0.2511, 0.1093])


In [ ]:
# 문백 벡터 계싼

# values = inputs @ W_value # V: (6, 3) @ (3, 2) → (6, 2)
context_vec_2 = attn_weights_2 @ values # (6,) @ (6, 2) → (2,)
print(context_vec_2)

tensor([0.2889, 0.4179])


In [21]:
#  셀프 어텐션 파이썬 클래스 구현하기

import torch
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # 예: d_in = 3, d_out = 2

        # W_query.shape = (3, 2)
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))

        # W_key.shape = (3, 2)
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))

        # W_value.shape = (3, 2)
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # x.shape = (토큰 개수, d_in) 예: (6, 3)
        # (6, 3) @ (3, 2) -> (6, 2)
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        # (6, 2) @ (2, 6) -> (6, 6)
        # 모든 Query와 모든 Key 사이의 점수
        attn_scores = queries @ keys.T

        # 특정 Query 토큰이 6개 모든 Key 토큰을 얼마나 참고할지 나타내는 점수
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # (6, 6) @ (6, 2) -> (6, 2)
        context_vec = attn_weights @ values
        return context_vec


In [22]:
torch.manual_seed(123)

sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.3052, 0.8119],
        [0.3118, 0.8275],
        [0.3115, 0.8268],
        [0.3002, 0.8004],
        [0.2980, 0.7956],
        [0.3057, 0.8133]], grad_fn=<MmBackward0>)


In [23]:
# 코드 3-2 파이토치 Linear 층을 사용한 셀프 어텐션 클래스

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [24]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0755,  0.0707],
        [-0.0763,  0.0698],
        [-0.0763,  0.0698],
        [-0.0778,  0.0676],
        [-0.0782,  0.0670],
        [-0.0771,  0.0687]], grad_fn=<MmBackward0>)
